# Evaluation Results Viewer
Reads one or more `eval_results.json` files (produced by `tools/test_tracked.py`) and presents the data as pandas DataFrames.

In [ ]:
import hashlib
import json
import os

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.lines import Line2D

# ── Configuration ────────────────────────────────────────────────────────────
# One path or a list of paths. Each file has the same nested JSON structure.
RESULTS_FILES = [
    "/local/home/nkoefarago/mmpose/benchmark/results/20260610_coco_e2e.json",
    "/local/home/nkoefarago/mmpose/benchmark/results/20260610_coco_topdown.json",
    # "/local/home/nkoefarago/mmpose/benchmark/results/20260610_crowdpose_e2e.json",
    # "/local/home/nkoefarago/mmpose/benchmark/results/20260610_crowdpose_topdown.json",
    # "/local/home/nkoefarago/mmpose/benchmark/results/20260611_ochuman_e2e.json",
    # "/local/home/nkoefarago/mmpose/benchmark/results/20260611_ochuman_topdown.json",
    # "/local/home/nkoefarago/mmpose/benchmark/results/20260615_emdb-mini_e2e.json",
    # "/local/home/nkoefarago/mmpose/benchmark/results/20260615_emdb-mini_topdown.json",
    # "/local/home/nkoefarago/mmpose/benchmark/results/20260615_emdb-mini_post_processed_oneeuro.json",
    # "/local/home/nkoefarago/mmpose/benchmark/results/20260615_emdb-mini_post_processed_smoothnet_ws8.json",
    # "/local/home/nkoefarago/mmpose/benchmark/results/20260615_emdb-mini_post_processed_smoothnet_ws32.json",
]
# Examples:
# RESULTS_FILES = '/absolute/path/to/eval_results.json'
# RESULTS_FILES = [
#     '/path/to/eval_results.json',
#     '/path/to/other_eval_results.json',
# ]

if isinstance(RESULTS_FILES, str):
    RESULTS_FILES = [RESULTS_FILES]


def load_eval_results(path):
    with open(path) as f:
        raw = json.load(f)
    records = []
    for model_name, variants in raw.items():
        for variant, runs in variants.items():
            for run in runs:
                record = {
                    'model': model_name,
                    'variant': variant,
                    'timestamp': pd.Timestamp(run['timestamp']),
                    'config': run.get('config', ''),
                    'checkpoint': run.get('checkpoint', ''),
                    'post_processed': bool(run.get('post_processed', False)),
                    'source_file': path,
                }
                record.update(run.get('metrics', {}))
                records.append(record)
    return records


# ── Load & flatten ───────────────────────────────────────────────────────────
records = []
for path in RESULTS_FILES:
    if not os.path.isfile(path):
        raise FileNotFoundError(f'Results file not found: {path}')
    records.extend(load_eval_results(path))

df = pd.DataFrame(records)

# Derive all metric columns (everything after the fixed columns)
ID_COLS = ["model", "variant", "timestamp"]
META_COLS = ['config', 'checkpoint', 'post_processed', 'source_file']
METRIC_COLS = [c for c in df.columns if c not in (ID_COLS + META_COLS)]

DEFAULT_MARKER = 'o'
POSTPROC_MARKER = '*'

# Fixed plot color per known model family (matplotlib tab20); extend order when adding families.
_MODEL_COLOR_ORDER = [
    'DARK', 'HRFormer', 'HRNet', 'MSPN', 'PCT', 'PETR', 'RF-DETR-Pose', 'RSN',
    'RTMPose', 'Sapiens', 'SimCC', 'UDP', 'ViTPose', 'YOLO-Pose', 'YOLO26-Pose',
]
_model_cmap = plt.get_cmap('tab20')
MODEL_COLORS = {m: _model_cmap(i) for i, m in enumerate(_MODEL_COLOR_ORDER)}

_TEMPORAL_PREFIXES = ('emdb', 'temporal')


def metric_label(col: str) -> str:
    """Short label for a metric column (strip emdb/ or temporal/ prefix)."""
    for prefix in _TEMPORAL_PREFIXES:
        prefix_slash = f'{prefix}/'
        if col.startswith(prefix_slash):
            return col[len(prefix_slash):]
    return col


def resolve_temporal_metric_col(df, suffix: str, prefixes=_TEMPORAL_PREFIXES):
    """Return emdb/* or temporal/* column for a metric suffix, preferring populated columns."""
    candidates = [
        f'{prefix}/{suffix}' for prefix in prefixes
        if f'{prefix}/{suffix}' in df.columns
    ]
    if not candidates:
        return None
    populated = [c for c in candidates if df[c].notna().any()]
    return populated[0] if populated else candidates[0]


def resolve_temporal_metric_cols(df, suffixes, prefixes=_TEMPORAL_PREFIXES):
    """Resolve multiple temporal metric columns (emdb/* or temporal/*)."""
    cols = []
    for suffix in suffixes:
        col = resolve_temporal_metric_col(df, suffix, prefixes)
        if col is not None:
            cols.append(col)
    return cols


def color_for_model(model: str):
    """Fixed color for known families; stable hash fallback for others."""
    if model in MODEL_COLORS:
        return MODEL_COLORS[model]
    digest = hashlib.md5(model.encode()).hexdigest()
    n_reserved = len(_MODEL_COLOR_ORDER)
    idx = n_reserved + (int(digest, 16) % (_model_cmap.N - n_reserved))
    return _model_cmap(idx)


def scatter_runs(ax, sub, x_col, y_col, model, *, show_model_label=False):
    """Scatter points for one model; stars mark post-processed runs."""
    color = color_for_model(model)
    is_pp = sub['post_processed'].fillna(False).astype(bool)
    raw = sub[~is_pp]
    pp = sub[is_pp]
    if not raw.empty:
        ax.scatter(
            raw[x_col],
            raw[y_col],
            s=60,
            alpha=0.85,
            color=color,
            marker=DEFAULT_MARKER,
            label=model if show_model_label else None,
        )
    if not pp.empty:
        ax.scatter(
            pp[x_col],
            pp[y_col],
            s=140,
            alpha=0.85,
            color=color,
            marker=POSTPROC_MARKER,
            label=None,
        )


def postproc_shape_legend_handles():
    return [
        Line2D(
            [0], [0], marker=DEFAULT_MARKER, color='0.45', linestyle='None',
            markersize=8, label='raw',
        ),
        Line2D(
            [0], [0], marker=POSTPROC_MARKER, color='0.45', linestyle='None',
            markersize=12, label='post-processed',
        ),
    ]

print(
    f'Loaded {len(df)} evaluation run(s) from {len(RESULTS_FILES)} file(s) '
    f'| metrics: {METRIC_COLS}'
)

## All evaluation entries

In [ ]:
display_cols = ID_COLS + METRIC_COLS

all_entries = (
    df[display_cols]
    .sort_values(['model', 'variant', 'timestamp'])
    .reset_index(drop=True)
)

display(
    all_entries.style
    .format({c: '{:.4f}' for c in METRIC_COLS}, na_rep='—')
    .set_caption('All evaluation runs')
    .set_table_styles([{'selector': 'caption',
                        'props': [('font-size', '14px'), ('font-weight', 'bold')]}])
)

## Latest result per model / variant

In [ ]:
DISPLAY_COLS = ["model", "variant"] + METRIC_COLS

AP_AR_COLS = [c for c in METRIC_COLS if '/AP' in c or '/AR' in c]
OTHER_METRIC_COLS = [c for c in METRIC_COLS if c not in AP_AR_COLS]

fmt = {c: (lambda x: f'{x*100:.1f}' if pd.notna(x) else '—') for c in AP_AR_COLS}
fmt.update({c: '{:.4f}' for c in OTHER_METRIC_COLS})

latest = (
    df.sort_values('timestamp')
    .groupby(['model', 'variant'], sort=False)
    .last()
    .reset_index()
)[DISPLAY_COLS].sort_values(['model', 'variant']).reset_index(drop=True)

display(
    latest.style
    .format(fmt, na_rep='—')
    .set_caption('Latest result per model / variant')
    .set_table_styles([{'selector': 'caption',
                        'props': [('font-size', '14px'), ('font-weight', 'bold')]}])
)

## Best `coco/AP` run per model

In [ ]:
AP_COL = 'coco/AP'

if AP_COL not in df.columns:
    print(f"Column '{AP_COL}' not found in results. "
          "Available metric columns:", METRIC_COLS)
else:
    best_ap = (
        df.dropna(subset=[AP_COL])
        .sort_values(AP_COL, ascending=False)
        .groupby('model', sort=False)
        .first()
        .reset_index()
    )[ID_COLS + METRIC_COLS].sort_values('model').reset_index(drop=True)

    display(
        best_ap.style
        .format({c: '{:.4f}' for c in METRIC_COLS}, na_rep='—')
        .set_caption(f'Best {AP_COL} run per model (variant + timestamp shown)')
        .set_table_styles([{'selector': 'caption',
                            'props': [('font-size', '14px'), ('font-weight', 'bold')]}])
    )

## AP vs e2e FPS

In [ ]:
AP_COL = 'coco/AP'
FPS_COL = 'perf/e2e/fps'
# Set to a positive int to label only the N highest-AP points (0 = no point labels).
ANNOTATE_TOP_N = 0

missing = [c for c in (AP_COL, FPS_COL) if c not in df.columns]
if missing:
    print(f'Missing columns for plot: {missing}')
    print('Available metric columns:', METRIC_COLS)
else:
    plot_df = (
        df.sort_values('timestamp')
        .groupby(['model', 'variant', 'post_processed'], sort=False)
        .last()
        .reset_index()
        .dropna(subset=[AP_COL, FPS_COL])
    )

    if plot_df.empty:
        print(f'No rows with both {AP_COL} and {FPS_COL}.')
    else:
        models = plot_df['model'].unique()

        fig, ax = plt.subplots(figsize=(15, 9))
        for model in models:
            if model == 'Sapiens':
                continue
            sub = plot_df[plot_df['model'] == model]
            scatter_runs(ax, sub, FPS_COL, AP_COL, model, show_model_label=True)

        if ANNOTATE_TOP_N > 0:
            top = plot_df.nlargest(ANNOTATE_TOP_N, AP_COL)
            for _, row in top.iterrows():
                ax.annotate(
                    f"{row['variant']}",
                    (row[FPS_COL], row[AP_COL]),
                    xytext=(4, 4),
                    textcoords='offset points',
                    fontsize=7,
                    color=color_for_model(row['model']),
                )

        ax.set_xlabel('e2e FPS')
        ax.set_ylabel('coco/AP')
        ax.set_title('Precision vs end-to-end throughput (latest run per model / variant)')
        ax.grid(True, alpha=0.3)
        handles, labels = ax.get_legend_handles_labels()
        ax.legend(
            handles=handles + postproc_shape_legend_handles(),
            title='model',
            bbox_to_anchor=(1.02, 1),
            loc='upper left',
            fontsize=8,
            framealpha=0.9,
        )
        fig.tight_layout()

        # Variant names for each point (no overlap on the chart).
        display(
            plot_df[['model', 'variant', 'post_processed', FPS_COL, AP_COL]]
            .sort_values([AP_COL, FPS_COL], ascending=[False, False])
            .reset_index(drop=True)
            .style.format({AP_COL: '{:.3f}', FPS_COL: '{:.1f}'})
            .set_caption('Points on plot (hover-free lookup)')
       
        )
        plt.show()

## AR vs e2e FPS

In [ ]:
AR_COL = 'coco/AR'
FPS_COL = 'perf/e2e/fps'
# Set to a positive int to label only the N highest-AR points (0 = no point labels).
ANNOTATE_TOP_N = 0

missing = [c for c in (AR_COL, FPS_COL) if c not in df.columns]
if missing:
    print(f'Missing columns for plot: {missing}')
    print('Available metric columns:', METRIC_COLS)
else:
    plot_df = (
        df.sort_values('timestamp')
        .groupby(['model', 'variant', 'post_processed'], sort=False)
        .last()
        .reset_index()
        .dropna(subset=[AR_COL, FPS_COL])
    )

    if plot_df.empty:
        print(f'No rows with both {AR_COL} and {FPS_COL}.')
    else:
        models = plot_df['model'].unique()

        fig, ax = plt.subplots(figsize=(15, 9))
        for model in models:
            if model == 'Sapiens':
                continue
            sub = plot_df[plot_df['model'] == model]
            scatter_runs(ax, sub, FPS_COL, AR_COL, model, show_model_label=True)

        if ANNOTATE_TOP_N > 0:
            top = plot_df.nlargest(ANNOTATE_TOP_N, AR_COL)
            for _, row in top.iterrows():
                ax.annotate(
                    f"{row['variant']}",
                    (row[FPS_COL], row[AR_COL]),
                    xytext=(4, 4),
                    textcoords='offset points',
                    fontsize=7,
                    color=color_for_model(row['model']),
                )

        ax.set_xlabel('e2e FPS')
        ax.set_ylabel('coco/AR')
        ax.set_title('Recall vs end-to-end throughput (latest run per model / variant)')
        ax.grid(True, alpha=0.3)
        handles, labels = ax.get_legend_handles_labels()
        ax.legend(
            handles=handles + postproc_shape_legend_handles(),
            title='model',
            bbox_to_anchor=(1.02, 1),
            loc='upper left',
            fontsize=8,
            framealpha=0.9,
        )
        fig.tight_layout()

        # Variant names for each point (no overlap on the chart).
        display(
            plot_df[['model', 'variant', 'post_processed', FPS_COL, AR_COL]]
            .sort_values([AR_COL, FPS_COL], ascending=[False, False])
            .reset_index(drop=True)
            .style.format({AR_COL: '{:.3f}', FPS_COL: '{:.1f}'})
            .set_caption('Points on plot (hover-free lookup)')
       
        )
        plt.show()

## EMDB temporal metrics vs e2e FPS

In [ ]:
FPS_COL = 'perf/e2e/fps'
TEMPORAL_METRIC_SUFFIXES = ['bMPJAE', 'bMPJVE', 'tMPJAE', 'tMPJVE']
TEMPORAL_METRIC_COLS = resolve_temporal_metric_cols(df, TEMPORAL_METRIC_SUFFIXES)
# Set to a positive int to label only the N best (lowest) points per subplot (0 = no labels).
ANNOTATE_TOP_N = 0

if not TEMPORAL_METRIC_COLS:
    print('No temporal metric columns found (expected emdb/* or temporal/*).')
    print('Available metric columns:', METRIC_COLS)
elif FPS_COL not in df.columns:
    print(f'Missing column for plot: {FPS_COL}')
    print('Available metric columns:', METRIC_COLS)
else:
    plot_df = (
        df.sort_values('timestamp')
        .groupby(['model', 'variant', 'post_processed'], sort=False)
        .last()
        .reset_index()
        .dropna(subset=[FPS_COL])
    )

    if plot_df.empty:
        print(f'No rows with {FPS_COL}.')
    else:
        models = plot_df['model'].unique()

        fig, axes = plt.subplots(2, 2, figsize=(15, 12))
        axes = axes.flatten()

        for ax, metric_col in zip(axes, TEMPORAL_METRIC_COLS):
            sub_df = plot_df.dropna(subset=[metric_col])
            if sub_df.empty:
                ax.set_title(f'{metric_col} (no data)')
                ax.axis('off')
                continue

            for model in models:
                if model == 'Sapiens':
                    continue
                sub = sub_df[sub_df['model'] == model]
                if sub.empty:
                    continue
                scatter_runs(
                    ax,
                    sub,
                    FPS_COL,
                    metric_col,
                    model,
                    show_model_label=(metric_col == TEMPORAL_METRIC_COLS[0]),
                )

            if ANNOTATE_TOP_N > 0:
                top = sub_df.nsmallest(ANNOTATE_TOP_N, metric_col)
                for _, row in top.iterrows():
                    ax.annotate(
                        f"{row['variant']}",
                        (row[FPS_COL], row[metric_col]),
                        xytext=(4, 4),
                        textcoords='offset points',
                        fontsize=7,
                        color=color_for_model(row['model']),
                    )

            ax.set_xlabel('e2e FPS')
            ax.set_ylabel(metric_label(metric_col))
            ax.set_title(metric_label(metric_col))
            ax.grid(True, alpha=0.3)

        handles, labels = axes[0].get_legend_handles_labels()
        fig.legend(
            handles=handles + postproc_shape_legend_handles(),
            title='model',
            bbox_to_anchor=(1.02, 0.98),
            loc='upper left',
            fontsize=8,
            framealpha=0.9,
        )
        fig.suptitle(
            'Temporal metrics vs end-to-end throughput '
            '(latest run per model / variant)',
            y=1.02,
        )
        fig.tight_layout()

        lookup_cols = ['model', 'variant', 'post_processed', FPS_COL] + TEMPORAL_METRIC_COLS
        display(
            plot_df[lookup_cols]
            .dropna(subset=TEMPORAL_METRIC_COLS, how='all')
            .sort_values(['model', 'variant'])
            .reset_index(drop=True)
            .style.format(
                {c: '{:.4f}' for c in TEMPORAL_METRIC_COLS} | {FPS_COL: '{:.1f}'}
            )
            .set_caption('Points on plot (hover-free lookup)')
        )
        plt.show()

## bMPJVE vs e2e FPS

In [ ]:
METRIC_COL_NAME = resolve_temporal_metric_col(df, 'bMPJVE')
FPS_COL = 'perf/e2e/fps'
# Set to a positive int to label only the N best (lowest) points (0 = no point labels).
ANNOTATE_TOP_N = 0

if METRIC_COL_NAME is None:
    print('No bMPJVE column found (expected emdb/bMPJVE or temporal/bMPJVE).')
    print('Available metric columns:', METRIC_COLS)
elif FPS_COL not in df.columns:
    print(f'Missing column for plot: {FPS_COL}')
    print('Available metric columns:', METRIC_COLS)
else:
    plot_df = (
        df.sort_values('timestamp')
        .groupby(['model', 'variant', 'post_processed'], sort=False)
        .last()
        .reset_index()
        .dropna(subset=[METRIC_COL_NAME, FPS_COL])
    )

    if plot_df.empty:
        print(f'No rows with both {METRIC_COL_NAME} and {FPS_COL}.')
    else:
        models = plot_df['model'].unique()

        fig, ax = plt.subplots(figsize=(15, 9))
        for model in models:
            if model == 'Sapiens':
                continue
            sub = plot_df[plot_df['model'] == model]
            scatter_runs(ax, sub, FPS_COL, METRIC_COL_NAME, model, show_model_label=True)

        if ANNOTATE_TOP_N > 0:
            top = plot_df.nsmallest(ANNOTATE_TOP_N, METRIC_COL_NAME)
            for _, row in top.iterrows():
                ax.annotate(
                    f"{row['variant']}",
                    (row[FPS_COL], row[METRIC_COL_NAME]),
                    xytext=(4, 4),
                    textcoords='offset points',
                    fontsize=7,
                    color=color_for_model(row['model']),
                )

        ax.set_xlabel('e2e FPS')
        ax.set_ylabel(metric_label(METRIC_COL_NAME))
        ax.set_title(
            f'{metric_label(METRIC_COL_NAME)} vs end-to-end throughput '
            '(latest run per model / variant)'
        )
        ax.grid(True, alpha=0.3)
        handles, labels = ax.get_legend_handles_labels()
        ax.legend(
            handles=handles + postproc_shape_legend_handles(),
            title='model',
            bbox_to_anchor=(1.02, 1),
            loc='upper left',
            fontsize=8,
            framealpha=0.9,
        )
        fig.tight_layout()

        display(
            plot_df[['model', 'variant', 'post_processed', FPS_COL, METRIC_COL_NAME]]
            .sort_values([METRIC_COL_NAME, FPS_COL], ascending=[True, False])
            .reset_index(drop=True)
            .style.format({METRIC_COL_NAME: '{:.4f}', FPS_COL: '{:.1f}'})
            .set_caption('Points on plot (hover-free lookup)')
        )
        plt.show()

## EMDB keypoint trajectory over time

Loads a benchmark prediction export (`manifest.json` + `frames.json` from `benchmark_e2e.py`) and plots one keypoint's **x** and **y** image coordinates across frames in a sequence.

In [ ]:
import json
import os.path as osp

# ── Configuration ────────────────────────────────────────────────────────────
# Directory with manifest.json and frames.json (benchmark_e2e export).
PRED_DIR = (
    '/local/home/nkoefarago/mmpose/'
    'benchmark/predictions/20260622_emdb_topdown/ViTPose-small-rfdetr__postproc'
)
SEQUENCE = None  # None = first sequence in the bundle, or e.g. 'P1/14_outdoor_climb'
KEYPOINT = 'right_wrist'  # name from dataset_meta, or int index (0–16 for COCO-17)
SHOW_GT = True  # overlay matched ground-truth keypoint
MIN_SCORE = 0.0  # skip pred points below this keypoint score


def _load_prediction_bundle(pred_dir: str):
    manifest_path = osp.join(pred_dir, 'manifest.json')
    frames_path = osp.join(pred_dir, 'frames.json')
    if not osp.isfile(manifest_path) or not osp.isfile(frames_path):
        raise FileNotFoundError(
            f'Expected manifest.json and frames.json in {pred_dir}')
    with open(manifest_path, encoding='utf-8') as f:
        manifest = json.load(f)
    with open(frames_path, encoding='utf-8') as f:
        frames = json.load(f)
    return manifest, frames


def _emdb_sequence_name(img_path: str) -> str:
    """P1/14_outdoor_climb/images/00042.jpg -> P1/14_outdoor_climb."""
    parts = img_path.replace('\\', '/').split('/')
    if len(parts) >= 3 and parts[-2] == 'images':
        return '/'.join(parts[:-2])
    return osp.dirname(img_path)


def _emdb_frame_index(img_path: str) -> int:
    basename = osp.splitext(osp.basename(img_path))[0]
    if basename.startswith('image_'):
        return int(basename.split('_')[-1])
    return int(basename)


def _keypoint_index(name_or_idx, dataset_meta: dict) -> int:
    if isinstance(name_or_idx, int):
        return name_or_idx
    id2name = dataset_meta.get('keypoint_id2name', {})
    for idx, name in id2name.items():
        if name == name_or_idx:
            return int(idx)
    for idx, info in dataset_meta.get('keypoint_info', {}).items():
        if info.get('name') == name_or_idx:
            return int(idx)
    names = list(id2name.values()) or [
        info.get('name') for info in dataset_meta.get('keypoint_info', {}).values()
    ]
    raise ValueError(
        f'Keypoint {name_or_idx!r} not found. Available: {names}')


def _pick_instance_idx(instances, matches, role: str):
    idx_key = 'pred_idx' if role == 'pred' else 'gt_idx'
    if matches:
        return matches[0][idx_key]
    if instances:
        return 0
    return None


def _extract_keypoint_trajectory(
    frames,
    sequence: str,
    kpt_idx: int,
    show_gt: bool,
    min_score: float,
):
    rows = []
    for frame in frames:
        if _emdb_sequence_name(frame['img_path']) != sequence:
            continue

        matches = frame['metrics'].get('matches', [])
        pred_instances = frame['predictions']['instances']
        gt_instances = frame['ground_truth']['instances']

        pred_i = _pick_instance_idx(pred_instances, matches, 'pred')
        if pred_i is None:
            continue

        pred_kpt = pred_instances[pred_i]['keypoints'][kpt_idx]
        pred_score = pred_instances[pred_i].get('keypoint_scores', [1.0])[kpt_idx]
        if pred_score < min_score:
            continue

        gt_kpt = None
        if show_gt:
            gt_i = _pick_instance_idx(gt_instances, matches, 'gt')
            if gt_i is not None:
                gt_kpt = gt_instances[gt_i]['keypoints'][kpt_idx]

        rows.append({
            'frame': _emdb_frame_index(frame['img_path']),
            'pred_x': pred_kpt[0],
            'pred_y': pred_kpt[1],
            'pred_score': pred_score,
            'gt_x': gt_kpt[0] if gt_kpt is not None else None,
            'gt_y': gt_kpt[1] if gt_kpt is not None else None,
        })

    return pd.DataFrame(rows).sort_values('frame').reset_index(drop=True)


pred_dir = osp.abspath(PRED_DIR)
manifest, frames = _load_prediction_bundle(pred_dir)
dataset_meta = manifest.get('dataset_meta', {})
kpt_idx = _keypoint_index(KEYPOINT, dataset_meta)
kpt_name = (
    KEYPOINT if isinstance(KEYPOINT, str)
    else dataset_meta.get('keypoint_id2name', {}).get(str(KEYPOINT), str(KEYPOINT))
)

sequences = sorted({_emdb_sequence_name(f['img_path']) for f in frames})
if not sequences:
    raise ValueError(f'No frames found in {pred_dir}')

sequence = SEQUENCE or sequences[0]
if sequence not in sequences:
    raise ValueError(
        f'Sequence {sequence!r} not in bundle. Available ({len(sequences)}): '
        f'{sequences[:10]}{"..." if len(sequences) > 10 else ""}'
    )

traj = _extract_keypoint_trajectory(
    frames, sequence, kpt_idx, SHOW_GT, MIN_SCORE)

if traj.empty:
    print(f'No trajectory points for sequence {sequence!r} '
          f'(keypoint={kpt_name}, min_score={MIN_SCORE}).')
else:
    model_label = manifest.get('model_name', 'model')
    if manifest.get('model_variant'):
        model_label = f'{model_label}-{manifest["model_variant"]}'

    fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
    axes[0].plot(traj['frame'], traj['pred_x'], color='C3', label='prediction')
    if SHOW_GT and traj['gt_x'].notna().any():
        axes[0].plot(
            traj['frame'], traj['gt_x'], color='C0', linestyle='--', label='ground truth')
    axes[0].set_ylabel('x (px)')
    axes[0].set_title(f'{kpt_name} — horizontal position')
    axes[0].grid(True, alpha=0.3)
    axes[0].legend(loc='upper right')

    axes[1].plot(traj['frame'], traj['pred_y'], color='C3', label='prediction')
    if SHOW_GT and traj['gt_y'].notna().any():
        axes[1].plot(
            traj['frame'], traj['gt_y'], color='C0', linestyle='--', label='ground truth')
    axes[1].set_xlabel('frame index')
    axes[1].set_ylabel('y (px)')
    axes[1].set_title(f'{kpt_name} — vertical position')
    axes[1].grid(True, alpha=0.3)
    axes[1].legend(loc='upper right')

    fig.suptitle(
        f'{model_label} | {manifest.get("test_dataset", "emdb")} | {sequence} '
        f'({len(traj)} frames)',
        y=1.02,
    )
    fig.tight_layout()
    plt.show()

    display(
        traj.head(10).style.format({
            'pred_x': '{:.1f}',
            'pred_y': '{:.1f}',
            'pred_score': '{:.3f}',
            'gt_x': '{:.1f}',
            'gt_y': '{:.1f}',
        }).set_caption(f'First frames — {sequence}')
    )
    if len(sequences) > 1:
        print(f'Other sequences ({len(sequences)} total): {sequences}')